# Tarea 2 — Primer análisis en Google Colab
**Estudiante:** Andres Mauricio Zambrano Londoño  
**Periodo:** 2026-2

## Pregunta
¿En qué medida las condiciones temporales y climáticas ayudan a anticipar la cantidad diaria de alquileres de bicicletas?

**Dataset:** Bike Sharing Dataset, archivo `day.csv` (731 días; el número final de filas y columnas se confirma al leerlo).  
**Fuente:** UCI Machine Learning Repository: https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset  
**Objetivo y:** `cnt`, total de alquileres diarios.  
**Problema:** describir la variación de la demanda diaria y estimar `cnt` a partir de variables de calendario y clima.

Es un conjunto pequeño, distinto de los datos de hurtos de Medellín, que permite practicar lectura, parseo, depuración, visualización y predicción. El análisis es exploratorio: no prueba causalidad ni garantiza generalización a otras ciudades.

## 1. Preparación del entorno
Se fija una semilla para reproducibilidad.

In [ ]:
import io, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)
pd.set_option("display.max_columns", None)
print("Semilla:", SEMILLA)

## 2. Lectura del dataset
El CSV se lee directamente desde el ZIP público de UCI; no depende de rutas locales ni de archivos temporales subidos a Colab.

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
with urllib.request.urlopen(URL, timeout=60) as response:
    zip_bytes = response.read()

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
    csv_name = next(n for n in zf.namelist() if n.lower().endswith("day.csv"))
    with zf.open(csv_name) as f:
        df = pd.read_csv(f, sep=",", encoding="utf-8",
                         na_values=["", "NA", "N/A", "null", "NULL"])

print("Archivo:", csv_name)
print("Filas y columnas después de leer:", df.shape)
display(df.head())

## 3. Parseo de tipos
Se convierte la fecha y se fuerzan explícitamente las columnas numéricas. Se cuentan los NaN nuevos que aparecen por errores de conversión.

In [ ]:
fecha_na_antes = df["dteday"].isna().sum()
df["dteday"] = pd.to_datetime(df["dteday"], format="%Y-%m-%d", errors="coerce")
print("NaN nuevos al parsear fecha:", int(df["dteday"].isna().sum() - fecha_na_antes))

numericas = ["instant","yr","mnth","holiday","weekday","workingday",
             "weathersit","temp","atemp","hum","windspeed","casual",
             "registered","cnt"]
na_parseo = {}
for col in numericas:
    na_antes = df[col].isna().sum()
    df[col] = pd.to_numeric(df[col], errors="coerce")
    na_parseo[col] = int(df[col].isna().sum() - na_antes)
display(pd.Series(na_parseo, name="NaN nuevos por parseo").to_frame())
display(df.dtypes.rename("tipo").to_frame())

## 4. Auditoría de faltantes, duplicados e imposibles
Se auditan los valores faltantes y las reglas de plausibilidad. No se eliminan datos solo por ser extremos.

In [ ]:
display(df.isna().sum().rename("faltantes").to_frame())
print("Duplicados completos:", int(df.duplicated().sum()))
print("Fechas repetidas:", int(df["dteday"].duplicated().sum()))

auditoria = {
 "fecha_invalida": df["dteday"].isna(),
 "temp_fuera_0_1": ~df["temp"].between(0,1),
 "atemp_fuera_0_1": ~df["atemp"].between(0,1),
 "hum_fuera_0_1": ~df["hum"].between(0,1),
 "windspeed_fuera_0_1": ~df["windspeed"].between(0,1),
 "conteos_negativos": (df[["casual","registered","cnt"]] < 0).any(axis=1),
 "cnt_no_es_suma": df["cnt"] != df["casual"] + df["registered"]
}
display(pd.Series({k:int(v.sum()) for k,v in auditoria.items()},
                  name="filas que incumplen").to_frame())
display(df.describe(include="all").T)

## 5. Decisiones de depuración
- Se eliminan duplicados completos para no contar dos veces la misma observación.
- Se eliminan fechas inválidas, mediciones climáticas fuera de [0, 1], conteos negativos y filas donde `cnt` no coincide con `casual + registered`.
- Los valores faltantes que permanezcan se imputan dentro del pipeline: mediana para variables numéricas y moda para categóricas, calculadas solo con entrenamiento.
- No se eliminan observaciones por ser simplemente atípicas.
- `casual` y `registered` no entran en X porque su suma es `cnt`: incluirlas sería filtración del objetivo.

In [ ]:
datos = df.drop_duplicates().copy()

# Los faltantes de predictores climáticos se conservan para imputarlos dentro del Pipeline.
# En cambio, la fecha y el objetivo cnt son indispensables para este análisis.
clima_valido = pd.Series(True, index=datos.index)
for col in ["temp", "atemp", "hum", "windspeed"]:
    clima_valido &= datos[col].isna() | datos[col].between(0, 1)

conteos_no_negativos = (
    (datos[["casual", "registered", "cnt"]].isna())
    | (datos[["casual", "registered", "cnt"]] >= 0)
).all(axis=1)

# Comprobar cnt = casual + registered solo cuando los tres valores están disponibles.
componentes_disponibles = datos[["casual", "registered", "cnt"]].notna().all(axis=1)
suma_coherente = (~componentes_disponibles) | (
    datos["cnt"] == datos["casual"] + datos["registered"]
)

valido = (
    datos["dteday"].notna()
    & datos["cnt"].notna()
    & clima_valido
    & conteos_no_negativos
    & suma_coherente
)
datos = datos.loc[valido].sort_values("dteday").reset_index(drop=True)
print("Antes:", df.shape, "| Después:", datos.shape)
print("Los faltantes de predictores que permanecen se imputan dentro del Pipeline.")
display(datos.isna().sum().rename("faltantes restantes").to_frame())

## 6. Gráficos
Se presentan dos figuras cuyos títulos están formulados como preguntas.

In [ ]:
plt.figure(figsize=(11,5))
plt.plot(datos["dteday"], datos["cnt"], linewidth=1.3)
plt.title("¿Cómo cambia la cantidad diaria de alquileres a lo largo del tiempo?")
plt.xlabel("Fecha"); plt.ylabel("Alquileres diarios")
plt.grid(alpha=.25); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8,5))
datos.boxplot(column="cnt", by="weathersit", grid=False)
plt.title("¿La cantidad de alquileres cambia según la situación climática?")
plt.suptitle("")
plt.xlabel("Situación climática (código)"); plt.ylabel("Alquileres diarios")
plt.tight_layout(); plt.show()

### Lectura de los gráficos
La serie temporal permite observar variaciones de demanda durante el periodo. El diagrama de cajas compara la distribución de alquileres según la categoría climática. Las diferencias visuales indican asociaciones posibles, pero no prueban causalidad: también intervienen calendario, temporada y otros factores.

## 7. Predicción: objetivo y variables
**y:** predecir el total de alquileres diarios (`cnt`).  
**X:** temporada, año, mes, festivo, día de semana, día laboral, situación climática, temperatura, sensación térmica, humedad y velocidad del viento.  
Se excluyen `casual` y `registered` porque componen exactamente `cnt`. También se excluyen `instant` (identificador secuencial) y `dteday` (fecha usada para ordenar y graficar).

In [ ]:
categoricas = ["season","yr","mnth","holiday","weekday","workingday","weathersit"]
numericas_modelo = ["temp","atemp","hum","windspeed"]
X = datos[categoricas + numericas_modelo].copy()
y = datos["cnt"].copy()
print("X:", X.shape, "| y:", y.shape)
print("Columnas predictoras:", list(X.columns))

## 8. Split y baseline
Se reserva el 20 % para prueba. El baseline predice la media de `y` calculada en el conjunto de entrenamiento. La métrica principal será MAE.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEMILLA
)
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
pred_base = baseline.predict(X_test)
mae_base = mean_absolute_error(y_test, pred_base)
print("Entrenamiento:", len(X_train), "| Prueba:", len(X_test))
print(f"MAE baseline: {mae_base:,.2f} alquileres")

## 9. Modelo de regresión lineal
El preprocesamiento se integra en un Pipeline para evitar que la información de prueba influya en la imputación, codificación o escalado.

In [ ]:
preproceso = ColumnTransformer([
    ("cat", Pipeline([
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categoricas),
    ("num", Pipeline([
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", StandardScaler())
    ]), numericas_modelo)
])
modelo = Pipeline([
    ("preproceso", preproceso),
    ("regresion", LinearRegression())
])
modelo.fit(X_train, y_train)
pred = modelo.predict(X_test)

mae_modelo = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred)
display(pd.DataFrame({
    "Modelo":["Baseline (media)","Regresión lineal"],
    "MAE":[mae_base,mae_modelo]
}))
print(f"RMSE regresión lineal: {rmse:,.2f}")
print(f"R² regresión lineal: {r2:.3f}")
print(f"Cambio porcentual del MAE vs baseline: {(mae_base-mae_modelo)/mae_base*100:.2f}%")

## 10. Evaluación y límites
MAE expresa el error absoluto medio en número de alquileres. Si el MAE de la regresión lineal es menor que el del baseline, el modelo mejora la referencia simple en esta partición; si es igual o mayor, no demuestra mejora.

La partición es aleatoria y no una validación temporal estricta, por lo que no debe interpretarse como garantía de desempeño futuro. El análisis identifica asociaciones, no causas, y el dataset corresponde a un sistema y periodo específicos: no se puede generalizar automáticamente a Medellín.

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(y_test, pred, alpha=.75)
lim_inf = min(y_test.min(), pred.min())
lim_sup = max(y_test.max(), pred.max())
plt.plot([lim_inf,lim_sup],[lim_inf,lim_sup],"--")
plt.title("¿Qué tan cerca quedan las predicciones de los valores reales?")
plt.xlabel("Alquileres reales"); plt.ylabel("Alquileres predichos")
plt.grid(alpha=.25); plt.tight_layout(); plt.show()

if mae_modelo < mae_base:
    print("La regresión lineal mejora el baseline según MAE en esta partición.")
else:
    print("La regresión lineal no mejora el baseline según MAE en esta partición.")

## 11. Conclusión
Este notebook ejecuta lectura, parseo, auditoría, depuración, visualización y predicción sobre un dataset tabular distinto de los registros de hurtos de Medellín. Se comparó una regresión lineal con un baseline de media mediante MAE.

**Completar tras ejecutar:** escribir aquí si la regresión lineal mejoró al baseline y cuál fue el MAE obtenido. No afirmar causalidad ni desempeño perfecto.

## 12. Referencias
- Fanaee-T, H. (2013). *Bike Sharing Dataset*. UCI Machine Learning Repository. https://doi.org/10.24432/C5W894
- pandas: https://pandas.pydata.org/docs/
- scikit-learn: https://scikit-learn.org/stable/